# Module 4: Advanced Geospatial Emoji Sentiment Analysis (LSTMs)

In this notebook, we combine **Natural Language Processing (NLP)** with **Geospatial Context** to create a powerful sentiment analysis tool. This technique is used by urban planners, disaster response teams, and brands to understand *what* people are saying and *where* they are saying it.

### The 7-Class Emoji Sentiment Scale
We've expanded the sentiment scale to include more nuanced emotions:
1.  😡 **Extremely Negative** (Class 0)
2.  😠 **Strongly Negative** (Class 1)
3.  😟 **Negative** (Class 2)
4.  😐 **Neutral** (Class 3)
5.  🙂 **Positive** (Class 4)
6.  😍 **Strongly Positive** (Class 5)
7.  🥰 **Extremely Positive** (Class 6)

### Geospatial NLP Applications
- **Crisis Mapping:** Identifying regions where residents are reporting infrastructure damage during a storm.
- **Public Sentiment:** Tracking how sentiment towards a new city park varies across different neighborhoods.
- **Logistics:** Monitoring delivery satisfaction in specific geographic zones.
- **Tourism:** Analyzing visitor feedback across different attractions in a city.

## 1. Data Preprocessing with Location

We will create an expanded synthetic dataset that includes text (reviews/tweets), a sentiment label (0-6), and location data (Latitude and Longitude) from various cities around the world.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import re
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd
import contextily as ctx

# Emoji Mapping for visualization
sentiment_map = {
    0: "😡",
    1: "😠", 
    2: "😟",
    3: "😐",
    4: "🙂",
    5: "😍",
    6: "🥰"
}

# Expanded Geospatial Dataset
# (Text, Sentiment Label, Lat, Lon, City, Country)
raw_data = [
    ("The local park is absolutely disgusting and dangerous.", 0, 40.7128, -74.0060, "New York", "USA"),
    ("I really hate the traffic in this part of town, it's terrible.", 1, 40.7580, -73.9855, "New York", "USA"),
    ("Just walking through the city, fairly standard day.", 3, 40.7306, -73.9352, "New York", "USA"),
    ("Had a great coffee near the museum, very pleasant!", 4, 34.0522, -118.2437, "Los Angeles", "USA"),
    ("The weather here is perfect! I am so happy in this city!", 5, 34.0928, -118.3287, "Los Angeles", "USA"),
    ("Garbage everywhere, nobody cleans this street.", 2, 34.0407, -118.2688, "Los Angeles", "USA"),
    ("It's okay, but moving here was a mistake.", 2, 51.5074, -0.1278, "London", "UK"),
    ("The architecture is alright, nothing special.", 3, 51.5300, -0.1100, "London", "UK"),
    ("Wonderful afternoon at the gallery.", 5, 51.5033, -0.1195, "London", "UK"),
    ("This city is absolutely magical! I'm in love with everything!", 6, 48.8566, 2.3522, "Paris", "France"),
    ("The food here is amazing! Best experience ever!", 6, 48.8611, 2.3364, "Paris", "France"),
    ("The service was terrible and the food was cold.", 1, 48.8667, 2.3333, "Paris", "France"),
    ("Beautiful architecture but very crowded.", 3, 52.5200, 13.4050, "Berlin", "Germany"),
    ("Great public transport and friendly people.", 4, 52.5115, 13.4020, "Berlin", "Germany"),
    ("Too expensive and not very clean.", 2, 52.5300, 13.3800, "Berlin", "Germany"),
    ("Amazing beaches and perfect weather!", 6, 41.3851, 2.1734, "Barcelona", "Spain"),
    ("The city is very beautiful but the pickpockets are a problem.", 2, 41.3902, 2.1575, "Barcelona", "Spain"),
    ("Great food and vibrant atmosphere.", 5, 41.3874, 2.1686, "Barcelona", "Spain"),
    ("Tokyo is incredible! So much to see and do.", 6, 35.6762, 139.6503, "Tokyo", "Japan"),
    ("The public transport is amazing and very efficient.", 5, 35.6895, 139.6917, "Tokyo", "Japan"),
    ("Too crowded and expensive.", 2, 35.7101, 139.8107, "Tokyo", "Japan"),
    ("Sydney is beautiful! The harbour is stunning.", 6, -33.8688, 151.2093, "Sydney", "Australia"),
    ("Great weather and friendly people.", 5, -33.8708, 151.2073, "Sydney", "Australia"),
    ("Too expensive and the traffic is bad.", 2, -33.8651, 151.2094, "Sydney", "Australia"),
    ("The city is very clean and well organized.", 4, 1.3521, 103.8198, "Singapore", "Singapore"),
    ("Great food and shopping.", 5, 1.3573, 103.8198, "Singapore", "Singapore"),
    ("Too hot and humid.", 2, 1.3438, 103.8085, "Singapore", "Singapore"),
    ("Amazing cultural experience!", 6, 28.6139, 77.2090, "New Delhi", "India"),
    ("The food is incredible!", 6, 28.6270, 77.2090, "New Delhi", "India"),
    ("Too crowded and polluted.", 1, 28.6358, 77.2246, "New Delhi", "India"),
    ("Beautiful city with a rich history.", 5, 41.0082, 28.9784, "Istanbul", "Turkey"),
    ("Great food and friendly people.", 5, 41.0151, 28.9795, "Istanbul", "Turkey"),
    ("Too crowded and traffic is terrible.", 2, 41.0219, 28.9784, "Istanbul", "Turkey"),
    ("Stunning architecture and great food.", 5, 45.4642, 9.1900, "Milan", "Italy"),
    ("Fashion capital of the world!", 6, 45.4668, 9.1900, "Milan", "Italy"),
    ("Too expensive and crowded.", 2, 45.4700, 9.1950, "Milan", "Italy"),
    ("Amazing city with great food and wine.", 6, 41.9028, 12.4964, "Rome", "Italy"),
    ("Stunning historical sites.", 5, 41.9075, 12.4823, "Rome", "Italy"),
    ("Too crowded and hot.", 2, 41.8902, 12.4922, "Rome", "Italy"),
    ("Beautiful beaches and clear water.", 6, 25.7617, -80.1918, "Miami", "USA"),
    ("Great weather and nightlife.", 5, 25.7743, -80.1937, "Miami", "USA"),
    ("Too crowded and expensive.", 2, 25.7814, -80.2166, "Miami", "USA"),
    ("Incredible natural beauty!", 6, 47.6062, -122.3321, "Seattle", "USA"),
    ("Great coffee and food scene.", 5, 47.6101, -122.3358, "Seattle", "USA"),
    ("Too much rain.", 2, 47.6134, -122.3442, "Seattle", "USA")
]

# Vocabulary Building
def tokenize(text):
    return re.sub(r"[^a-zA-Z0-9\s]", "", text.lower()).split()

all_tokens = []
for text, label, lat, lon, city, country in raw_data: 
    all_tokens.extend(tokenize(text))

vocab = {word: i+2 for i, (word, c) in enumerate(Counter(all_tokens).most_common())}
vocab["<PAD>"], vocab["<UNK>"] = 0, 1
inv_vocab = {i: word for word, i in vocab.items()}

print(f"Vocabulary Size: {len(vocab)}")
print(f"Number of samples: {len(raw_data)}")

## 2. Multi-Class LSTM Dataset
We need to handle the integer labels (0-6) and pad the sequences for training.

In [ ]:
MAX_LEN = 15

class GeoEmojiDataset(Dataset):
    def __init__(self, data, vocab, max_len):
        self.data = data
        self.vocab = vocab
        self.max_len = max_len
        
    def __len__(self): 
        return len(self.data)
        
    def __getitem__(self, idx):
        text, label, lat, lon, city, country = self.data[idx]
        indices = [self.vocab.get(t, 1) for t in tokenize(text)][:self.max_len]
        indices += [0] * (self.max_len - len(indices))
        return torch.tensor(indices), torch.tensor(label, dtype=torch.long)

dataset = GeoEmojiDataset(raw_data, vocab, MAX_LEN)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

sample_x, sample_y = next(iter(loader))
print(f"Target labels for sample batch: {sample_y.tolist()}")
print(f"Sentiment emojis for sample batch: {[sentiment_map[label] for label in sample_y.tolist()]}")

## 3. Deep Learning Architecture
Our model now outputs **7 values** representing the scores for each sentiment emoji.

In [ ]:
class GeoSentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=7):
        super(GeoSentimentLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, num_layers=2, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        return self.fc(hidden[-1])

model = GeoSentimentLSTM(len(vocab), 64, 128)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Model set for 7-class emoji classification.")

## 4. Training the Model

In [ ]:
epochs = 50
loss_history = []

for epoch in range(epochs):
    total_loss = 0
    for x, y in loader:
        optimizer.zero_grad()
        y_pred = model(x)
        loss = criterion(y_pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    
    epoch_loss = total_loss / len(dataset)
    loss_history.append(epoch_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}")

# Plot loss history
plt.figure(figsize=(10, 4))
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Over Time')
plt.grid(True, alpha=0.3)
plt.show()

## 5. Geospatial Visualization Dashboard
We will map our predictions onto a coordinate space to see regional sentiment trends using geopandas for better visualization.

In [ ]:
def map_sentiment(data):
    model.eval()
    
    # Create a DataFrame for geopandas
    df = pd.DataFrame(data, columns=['text', 'label', 'lat', 'lon', 'city', 'country'])
    predictions = []
    emojis = []

    for text in df['text']:
        tokens = tokenize(text)
        indices = [vocab.get(t, 1) for t in tokens][:MAX_LEN]
        indices += [0] * (MAX_LEN - len(indices))
        
        with torch.no_grad():
            pred = model(torch.tensor([indices]))
            pred_class = torch.argmax(pred, 1).item()
            emoji = sentiment_map[pred_class]
        
        predictions.append(pred_class)
        emojis.append(emoji)

    df['prediction'] = predictions
    df['emoji'] = emojis

    # Create geometry from lat/lon
    df['geometry'] = df.apply(lambda row: Point(row['lon'], row['lat']), axis=1)
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')

    # Create plot
    fig, ax = plt.subplots(figsize=(15, 10))

    # Plot each point with emoji
    for idx, row in gdf.iterrows():
        ax.text(row['lon'], row['lat'], row['emoji'], fontsize=20, ha='center', va='center')
        ax.annotate(f"{row['city']}, {row['country']}\n'{row['text'][:25]}...'", 
                    (row['lon'], row['lat']), 
                    xytext=(5, 5), 
                    textcoords='offset points', 
                    fontsize=9, 
                    bbox=dict(facecolor='white', alpha=0.7, pad=2))

    # Add basemap
    gdf_web = gdf.to_crs('EPSG:3857')
    ax = gdf_web.plot(ax=ax, markersize=0)
    ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)
    
    # Set plot boundaries and labels
    ax.set_xlim(-125, 155)
    ax.set_ylim(-40, 60)
    ax.set_xlabel("Longitude", fontsize=12)
    ax.set_ylabel("Latitude", fontsize=12)
    ax.set_title("Global Sentiment Snapshot (7-Class Emoji-Scale Map)", fontsize=14)
    ax.grid(True, alpha=0.3)

    plt.show()

    # Return the GeoDataFrame for further analysis
    return gdf

# Generate the map
sentiment_gdf = map_sentiment(raw_data)

## 6. Sentiment Analysis by Country
Let's analyze the average sentiment for each country in our dataset.

In [ ]:
# Calculate average sentiment per country
country_sentiment = sentiment_gdf.groupby('country')['prediction'].mean().sort_values()

# Create sentiment distribution plot
plt.figure(figsize=(12, 6))

# Create a color map for sentiment values
colors = []
for sentiment in country_sentiment:
    if sentiment < 2:
        colors.append('#FF6B6B')  # Red for negative
    elif sentiment < 4:
        colors.append('#FFD93D')  # Yellow for neutral
    else:
        colors.append('#6BCB77')  # Green for positive

country_sentiment.plot(kind='bar', color=colors)

plt.title('Average Sentiment by Country', fontsize=14)
plt.xlabel('Country', fontsize=12)
plt.ylabel('Average Sentiment Score', fontsize=12)
plt.xticks(rotation=45, ha='right')

# Add emoji labels to the plot
for i, (country, score) in enumerate(country_sentiment.items()):
    emoji = sentiment_map[round(score)]
    plt.text(i, score + 0.1, emoji, fontsize=16, ha='center')

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("Average sentiment scores per country:")
for country, score in country_sentiment.items():
    emoji = sentiment_map[round(score)]
    print(f"{country}: {score:.2f} ({emoji})")

## 7. Real-time Inference
Input a phrase and see which emoji the model assigns!

In [ ]:
def predict_emoji(phrase):
    model.eval()
    indices = [vocab.get(t, 1) for t in tokenize(phrase)][:MAX_LEN]
    indices += [0] * (MAX_LEN - len(indices))
    pred = model(torch.tensor([indices]))
    pred_class = torch.argmax(pred, 1).item()
    emoji = sentiment_map[pred_class]
    
    # Display confidence scores for each sentiment class
    print(f"\nPhrase: '{phrase}'")
    print(f"Predicted Sentiment: {emoji} (Class {pred_class})")
    print("\nConfidence Scores:")
    
    # Calculate probabilities
    probabilities = torch.nn.functional.softmax(pred, dim=1).tolist()[0]
    for i, prob in enumerate(probabilities):
        print(f"  {sentiment_map[i]}: {prob:.2f}")
    
    return pred_class, emoji, probabilities

# Test the model with some examples
predict_emoji("I love this amazing sunny morning in London")
predict_emoji("Total disaster area, truly horrible experience")
predict_emoji("The food was okay but the service was slow")
predict_emoji("This city is absolutely magical! I'm in love with everything!")

## 8. Model Evaluation
Let's evaluate the model's performance on our dataset.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Get all predictions
model.eval()
all_predictions = []
all_labels = []

for text, label, lat, lon, city, country in raw_data:
    tokens = tokenize(text)
    indices = [vocab.get(t, 1) for t in tokens][:MAX_LEN]
    indices += [0] * (MAX_LEN - len(indices))
    
    with torch.no_grad():
        pred = model(torch.tensor([indices]))
        pred_class = torch.argmax(pred, 1).item()
        
    all_predictions.append(pred_class)
    all_labels.append(label)

# Print classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_predictions, 
                           target_names=[sentiment_map[i] for i in range(7)]))

# Create confusion matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(all_labels, all_predictions)
sns.heatmap(cm, 
            annot=True, 
            fmt='d', 
            cmap='Blues',
            xticklabels=[sentiment_map[i] for i in range(7)],
            yticklabels=[sentiment_map[i] for i in range(7)])
plt.title('Confusion Matrix', fontsize=14)
plt.xlabel('Predicted Sentiment', fontsize=12)
plt.ylabel('True Sentiment', fontsize=12)
plt.tight_layout()
plt.show()

## 9. Save and Load Model
Let's save our trained model for future use.

In [ ]:
# Save the model
torch.save(model.state_dict(), 'geo_emoji_sentiment_model.pt')
print("Model saved successfully.")

# Save the vocabulary
import pickle
with open('geo_emoji_sentiment_vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)
print("Vocabulary saved successfully.")

# To load the model later:
# model = GeoSentimentLSTM(len(vocab), 64, 128)
# model.load_state_dict(torch.load('geo_emoji_sentiment_model.pt'))
# model.eval()

# To load the vocabulary later:
# with open('geo_emoji_sentiment_vocab.pkl', 'rb') as f:
#     vocab = pickle.load(f)

## 10. Conclusion

### Key Achievements:
1. **7-Class Emoji Sentiment Scale**: Enhanced sentiment classification from binary/ternary to 7 nuanced emotional states
2. **Geospatial Visualization**: Combined NLP with location data to map sentiment across the globe
3. **Deep Learning**: Implemented an LSTM-based model for multi-class sentiment analysis
4. **Basemap Integration**: Added OpenStreetMap basemap for improved context and readability
5. **Country-wise Analysis**: Calculated and visualized average sentiment scores per country
6. **Interactive Inference**: Created real-time prediction function with emoji outputs
7. **Model Evaluation**: Assessed performance using classification reports and confusion matrices

### Applications of This Work:
- **Urban Planning**: Identify neighborhoods with low sentiment for targeted improvements
- **Tourism**: Analyze visitor feedback across different cities and attractions
- **Brand Reputation**: Monitor customer sentiment in specific geographic regions
- **Crisis Response**: Track sentiment during emergencies to prioritize aid distribution
- **Real Estate**: Evaluate sentiment trends in different housing markets

This approach demonstrates how combining natural language processing with geospatial data can provide valuable insights for decision-making across various domains.